In [ ]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append('..')

In [ ]:
import numpy as np
import pylab as plt

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

In [ ]:
from vimms.Common import POSITIVE
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.features import obs_to_dfs
from vimms_gym.evaluation import evaluate
from vimms_gym.policy import fullscan_policy, random_policy, topN_policy, best_ppo_policy

# 1. Parameters

In [ ]:
# n_chemicals = (200, 500)
# mz_range = (100, 600)
# rt_range = (0, 300)
# intensity_range = (1E5, 1E10)

In [ ]:
n_chemicals = (500, 2000)
mz_range = (100, 600)
rt_range = (200, 1000)
intensity_range = (1E5, 1E10)

In [ ]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [ ]:
isolation_window = 0.7
N = 10
rt_tol = 120
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE
noise_density = 0.3
noise_max_val = 1e4

In [ ]:
mzml_filename = 'Beer_multibeers_1_fullscan1.mzML'
mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                       min_log_intensity=min_log_intensity,
                                       max_log_intensity=max_log_intensity)

In [ ]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
    },
    'noise': {
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [ ]:
max_peaks = 20
out_dir = 'results_%d_%d' % (max_peaks, rt_tol)

In [ ]:
total_timesteps = 2E6
n_eval_episodes = 10
deterministic = True

# 2. Training

## a. PPO

In [ ]:
model_name = 'PPO'

In [ ]:
# default parameters
learning_rate = 0.0003
batch_size = 64
n_steps = 2048
ent_coef = 0.0
gamma = 0.99
gae_lambda = 0.95

# modified parameters
# learning_rate = 0.00001
# batch_size = 128
# n_steps = 256
# ent_coef = 0.01
# gamma = 0.1
# gae_lambda = 0.7

In [ ]:
env = DDAEnv(max_peaks, params)
check_env(env)

In [ ]:
tensorboard_log = './%s/%s_DDAEnv_tensorboard' % (out_dir, model_name)

model = PPO("MultiInputPolicy", env, 
            tensorboard_log=tensorboard_log, 
            learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
            ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
model.learn(total_timesteps=total_timesteps)

In [ ]:
# model = PPO("MultiInputPolicy", env, verbose=2, 
#             learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
#             ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda)
# model.learn(total_timesteps=total_timesteps, log_interval=1)

In [ ]:
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=n_eval_episodes, 
                                          deterministic=deterministic)
mean_reward, std_reward

In [ ]:
fname = '%s/model_%s' % (out_dir, model_name)
model.save(fname)

## b. DQN

In [ ]:
model_name = 'DQN'

In [ ]:
# default parameters
learning_rate = 0.0001
batch_size = 32
gamma = 0.99
exploration_fraction = 0.1
exploration_initial_eps = 1.0
exploration_final_eps = 0.05

# modified parameters
# learning_rate = 0.0003
# batch_size = 32
# gamma = 0.8
# exploration_fraction = 0.25
# exploration_initial_eps = 1.0
# exploration_final_eps = 0.10

In [ ]:
env = DDAEnv(max_peaks, params)
check_env(env)

In [ ]:
tensorboard_log = './%s/%s_DDAEnv_tensorboard' % (out_dir, model_name)

model = DQN("MultiInputPolicy", env, 
            exploration_fraction=exploration_fraction, exploration_initial_eps=exploration_initial_eps, exploration_final_eps=exploration_final_eps, 
            tensorboard_log=tensorboard_log, 
            learning_rate=learning_rate, batch_size=batch_size, gamma=gamma)
model.learn(total_timesteps=total_timesteps)

In [ ]:
# model = DQN("MultiInputPolicy", env, 
#             exploration_fraction=exploration_fraction, exploration_initial_eps=exploration_initial_eps, exploration_final_eps=exploration_final_eps, 
#             learning_rate=learning_rate, batch_size=batch_size, gamma=gamma, verbose=2)
# model.learn(total_timesteps=total_timesteps, log_interval=1)

In [ ]:
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=n_eval_episodes, 
                                          deterministic=deterministic)
mean_reward, std_reward

In [ ]:
fname = '%s/model_%s' % (out_dir, model_name)
model.save(fname)